In [1]:
import pandas as pd
import sqlite3
import os

project_path = os.path.join(os.path.expanduser("~"), "Documents", "Healthcare_SQL_Analytics")
db_path = os.path.join(project_path, "healthcare_readmissions.db")
exports_path = os.path.join(project_path, "outputs", "exports")

os.makedirs(exports_path, exist_ok=True)   # make outputs/exports/ if it doesn't exist yet

conn = sqlite3.connect(db_path)

def run(sql):
    return pd.read_sql(sql, conn)

print("Exports will be saved to:", exports_path)

Exports will be saved to: C:\Users\lbona\Documents\Healthcare_SQL_Analytics\outputs\exports


In [2]:
inpatient = run("""
SELECT
    CASE
        WHEN n_inpatient = 0 THEN '0 prior stays'
        WHEN n_inpatient = 1 THEN '1 prior stay'
        WHEN n_inpatient BETWEEN 2 AND 3 THEN '2-3 prior stays'
        ELSE '4+ prior stays'
    END AS prior_inpatient_group,
    COUNT(*) AS encounters,
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
FROM fact_encounters
GROUP BY prior_inpatient_group
ORDER BY MIN(n_inpatient);
""")

inpatient.to_csv(os.path.join(exports_path, "readmit_by_prior_inpatient.csv"), index=False)
inpatient

,prior_inpatient_group,encounters,readmit_pct
0,0 prior stays,16537,39.9
1,1 prior stay,4926,54.7
2,2-3 prior stays,2742,66.4
3,4+ prior stays,795,81.3


In [3]:
exports = {
    "readmit_by_age": """
        SELECT a.age_band, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters f
        JOIN dim_age a ON f.age_band = a.age_band
        GROUP BY a.age_band, a.sort_order
        ORDER BY a.sort_order;
    """,
    "readmit_by_prior_er": """
        SELECT
            CASE WHEN n_emergency = 0 THEN '0 ER visits'
                 WHEN n_emergency = 1 THEN '1 ER visit'
                 ELSE '2+ ER visits' END AS prior_er_group,
            COUNT(*) AS encounters,
            ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters
        GROUP BY prior_er_group
        ORDER BY MIN(n_emergency);
    """,
    "readmit_by_a1c": """
        SELECT a1c_test, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters GROUP BY a1c_test ORDER BY readmit_pct DESC;
    """,
    "readmit_by_diabetes_med": """
        SELECT diabetes_med, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters GROUP BY diabetes_med ORDER BY readmit_pct DESC;
    """,
    "readmit_by_glucose": """
        SELECT glucose_test, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters GROUP BY glucose_test ORDER BY readmit_pct DESC;
    """,
    "readmit_by_primary_diagnosis": """
        SELECT d.diagnosis_category AS primary_diagnosis, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters f
        JOIN bridge_encounter_diagnosis b ON f.encounter_id = b.encounter_id
        JOIN dim_diagnosis d ON b.diagnosis_id = d.diagnosis_id
        WHERE b.diagnosis_position = 1
        GROUP BY d.diagnosis_category ORDER BY readmit_pct DESC;
    """,
    "readmit_by_specialty": """
        SELECT s.specialty_name, COUNT(*) AS encounters,
               ROUND(AVG(CASE WHEN f.readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS readmit_pct
        FROM fact_encounters f
        JOIN dim_specialty s ON f.specialty_id = s.specialty_id
        GROUP BY s.specialty_name ORDER BY readmit_pct DESC;
    """,
}

for name, query in exports.items():
    result = run(query)
    result.to_csv(os.path.join(exports_path, name + ".csv"), index=False)
    print("Saved:", name + ".csv", "(", len(result), "rows )")

Saved: readmit_by_age.csv ( 6 rows )
Saved: readmit_by_prior_er.csv ( 3 rows )
Saved: readmit_by_a1c.csv ( 3 rows )
Saved: readmit_by_diabetes_med.csv ( 2 rows )
Saved: readmit_by_glucose.csv ( 3 rows )
Saved: readmit_by_primary_diagnosis.csv ( 8 rows )
Saved: readmit_by_specialty.csv ( 7 rows )


In [4]:
summary = run("""
SELECT
    ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1) AS overall_readmit_pct,
    COUNT(*) AS total_encounters,
    (SELECT ROUND(AVG(CASE WHEN readmitted = 'yes' THEN 1.0 ELSE 0 END) * 100, 1)
     FROM fact_encounters
     WHERE n_inpatient >= 4) AS high_risk_readmit_pct
FROM fact_encounters;
""")

summary.to_csv(os.path.join(exports_path, "summary_kpis.csv"), index=False)
summary

,overall_readmit_pct,total_encounters,high_risk_readmit_pct
0,47.0,25000,81.3
